[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/02-census-block-groups.ipynb)

# Census Block Groups

A **census block group** is the smallest geographic unit for which the US Census Bureau publishes demographic data. Each block group typically contains 600–3,000 people.

In this notebook you will learn how to:

1. Find block groups inside an isochrone polygon
2. Inspect the block group data structure
3. Understand GEOID anatomy
4. Use point + radius lookup
5. Compare different radii
6. Build a summary table with pandas

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import create_isochrone, get_census_blocks

## 1. Block Groups from an Isochrone

First create an isochrone, then pass it to `get_census_blocks`.

In [ ]:
iso = create_isochrone("Austin, TX", travel_time=15, travel_mode="drive")
print(f"Isochrone area: {iso['properties']['area_sq_km']:.1f} sq km")

In [ ]:
blocks = get_census_blocks(polygon=iso)
print(f"Block groups found: {len(blocks)}")

## 2. Inspect the Block Group Structure

Each block group is a dict with FIPS codes, geometry, and area.

In [ ]:
sample = blocks[0]
for key, value in sample.items():
    if key == "geometry":
        print(f"{key}: <GeoJSON {value['type']} with {len(value['coordinates'][0])} vertices>")
    else:
        print(f"{key}: {value}")

## 3. GEOID Anatomy

The 12-digit GEOID encodes the full census hierarchy:

```
GEOID:  484530011041
        ││││││││││││
        SS           ← State FIPS  (2 digits) — e.g. 48 = Texas
          CCC         ← County FIPS (3 digits) — e.g. 453 = Travis County
             TTTTTT    ← Tract       (6 digits) — e.g. 001104
                   B   ← Block Group (1 digit)  — e.g. 1
```

In [ ]:
geoid = sample["geoid"]
print(f"Full GEOID:  {geoid}")
print(f"State FIPS:  {geoid[:2]}  (= sample['state_fips']: {sample['state_fips']})")
print(f"County FIPS: {geoid[2:5]}  (= sample['county_fips']: {sample['county_fips']})")
print(f"Tract:       {geoid[5:11]}  (= sample['tract']: {sample['tract']})")
print(f"Block Group: {geoid[11]}  (= sample['block_group']: {sample['block_group']})")

## 4. Point + Radius Lookup

Instead of a polygon you can pass `location=(lat, lon)` with a `radius_km`.

In [ ]:
# University of Texas campus
blocks_point = get_census_blocks(location=(30.2849, -97.7341), radius_km=3)
print(f"Block groups within 3 km of UT Austin: {len(blocks_point)}")

## 5. Compare Radii

See how the number of block groups grows with radius.

In [ ]:
radii = [1, 3, 5, 10]
for r in radii:
    result = get_census_blocks(location=(30.2849, -97.7341), radius_km=r)
    print(f"{r:>2} km radius: {len(result):>4} block groups")

## 6. Summary Table with Pandas

Build a quick overview of the block groups you retrieved.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "geoid": b["geoid"],
        "state_fips": b["state_fips"],
        "county_fips": b["county_fips"],
        "tract": b["tract"],
        "block_group": b["block_group"],
        "area_sq_km": round(b["area_sq_km"], 2),
    }
    for b in blocks
])

print(f"Total block groups: {len(df)}")
print(f"Unique counties: {df['county_fips'].nunique()}")
print(f"Unique tracts: {df['tract'].nunique()}")
print(f"Total area: {df['area_sq_km'].sum():.1f} sq km")
print()
df.head(10)

## Summary

| What you learned | API |
|---|---|
| Find block groups in a polygon | `get_census_blocks(polygon=iso)` |
| Find block groups around a point | `get_census_blocks(location=(lat, lon), radius_km=5)` |
| Parse the 12-digit GEOID | `SS` + `CCC` + `TTTTTT` + `B` |

**Next notebook:** [03 — Census Demographics](03-census-demographics.ipynb)